In [ ]:
# ============================================================================
# SNIPPET 1: MobileNetV2 - COMPLETE TRAINING, EVALUATION & RESULTS EXPORT
# ============================================================================
# This snippet handles:
# 1. Data loading (train/val/test)
# 2. MobileNetV2 model construction with transfer learning
# 3. Training on train set, monitoring on validation set
# 4. Full evaluation on TEST set with metrics & visualizations
# 5. Saving model + metadata for XAI pipelines
# ============================================================================

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (confusion_matrix, classification_report, 
                             precision_recall_fscore_support, accuracy_score)

# ============================================================================
# CONFIGURATION & PATHS
# ============================================================================
print("\n" + "="*100)
print(" " * 30 + "SHRIMP DISEASE CLASSIFICATION")
print(" " * 25 + "MobileNetV2 Transfer Learning Pipeline")
print("="*100)

# Dataset paths (using Dataset 2: Traditional Augmentation)
train_dir = '/kaggle/input/elsevier-disease-shrimp-data/2_Conventional_augmented_2500_ttvs/train'
validation_dir = '/kaggle/input/elsevier-disease-shrimp-data/2_Conventional_augmented_2500_ttvs/validation'
test_dir = '/kaggle/input/elsevier-disease-shrimp-data/2_Conventional_augmented_2500_ttvs/test'

# Hyperparameters
IMG_SIZE = (250, 250)
BATCH_SIZE = 64
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
DROPOUT_RATE = 0.3
DENSE_UNITS = 512




# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================
def format_time(seconds):
    """Convert seconds to readable time format"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    
    if hours > 0:
        return f"{hours}h {minutes}m {secs}s"
    elif minutes > 0:
        return f"{minutes}m {secs}s"
    else:
        return f"{secs}s"

def print_section(title):
    """Print formatted section header"""
    print("\n" + "="*100)
    print(f" {title}")
    print("="*100)

# ============================================================================
# DATA LOADING & PREPROCESSING
# ============================================================================
print_section("1. LOADING AND PREPARING DATA")

# Data augmentation for training
train_datagen = ImageDataGenerator(rescale=1./255)

# No augmentation for validation and test
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create data generators
print("\n📂 Creating data generators...")
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_gen = val_test_datagen.flow_from_directory(
    validation_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

test_gen = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Get class information
class_names = list(train_gen.class_indices.keys())
num_classes = len(class_names)
idx_to_class = {v: k for k, v in train_gen.class_indices.items()}

print(f"\n✓ Data generators created successfully")
print(f"  Class names: {class_names}")
print(f"  Number of classes: {num_classes}")
print(f"  Training samples: {train_gen.samples}")
print(f"  Validation samples: {val_gen.samples}")
print(f"  Test samples: {test_gen.samples}")

# ============================================================================
# MODEL CONSTRUCTION
# ============================================================================
print_section("2. BUILDING MobileNetV2 MODEL")

# Start timer
start_time_total = time.time()

print("\n🏗️  Constructing MobileNetV2 with transfer learning...")

# Load pre-trained MobileNetV2 (ImageNet weights)
base_model = MobileNetV2(
    input_shape=(250, 250, 3),
    include_top=False,
    weights='imagenet'
)

print(f"✓ MobileNetV2 base model loaded")
print(f"  Input shape: (250, 250, 3)")
print(f"  Include top: False (custom head will be added)")

# Freeze base model layers (transfer learning)
print("\n❄️  Freezing base model layers...")
for layer in base_model.layers:
    layer.trainable = False

frozen_layers = len([l for l in base_model.layers if not l.trainable])
trainable_layers = len([l for l in base_model.layers if l.trainable])
print(f"✓ Base model frozen")
print(f"  Frozen layers: {frozen_layers}")
print(f"  Trainable layers: {trainable_layers}")

# Add custom classification head
print("\n🔌 Adding custom classification head...")
x = GlobalAveragePooling2D()(base_model.output)
print(f"  ├─ GlobalAveragePooling2D()")
x = Dense(DENSE_UNITS, activation='relu')(x)
print(f"  ├─ Dense({DENSE_UNITS}, activation='relu')")
x = Dropout(DROPOUT_RATE)(x)
print(f"  ├─ Dropout({DROPOUT_RATE})")
predictions = Dense(num_classes, activation='softmax')(x)
print(f"  └─ Dense({num_classes}, activation='softmax')")

# Create final model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile model
print("\n⚙️  Compiling model...")
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(f"✓ Model compiled")
print(f"  Optimizer: Adam(lr={LEARNING_RATE})")
print(f"  Loss: categorical_crossentropy")
print(f"  Metrics: accuracy")

# Model summary
total_params = model.count_params()
print(f"\n📊 Model Summary:")
print(f"  Total parameters: {total_params:,}")
print(f"  Base model parameters: {base_model.count_params():,}")
print(f"  Custom head parameters: {total_params - base_model.count_params():,}")

# ============================================================================
# TRAINING
# ============================================================================
print_section("3. TRAINING MobileNetV2")

print(f"\n🚀 Starting training on {train_gen.samples} training samples...")
print(f"  Validation monitoring on {val_gen.samples} samples")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")

train_start = time.time()
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=NUM_EPOCHS,
    verbose=1
)
train_time = time.time() - train_start

print(f"\n✓ Training complete in {format_time(train_time)}")

# ============================================================================
# VISUALIZATION: TRAINING CURVES
# ============================================================================
print_section("4. TRAINING PERFORMANCE VISUALIZATION")

print("\n📈 Generating training/validation curves...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy plot
axes[0].plot(history.history['accuracy'], linewidth=2.5, label='Training', 
             color='#0066cc', marker='o', markersize=4, alpha=0.8)
axes[0].plot(history.history['val_accuracy'], linewidth=2.5, label='Validation', 
             color='#ff6600', marker='s', markersize=4, alpha=0.8)
axes[0].set_title('MobileNetV2 - Training vs Validation Accuracy', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=12, loc='lower right')
axes[0].grid(True, alpha=0.3, linestyle='--')
axes[0].set_ylim([0, 1])

# Loss plot
axes[1].plot(history.history['loss'], linewidth=2.5, label='Training', 
             color='#0066cc', marker='o', markersize=4, alpha=0.8)
axes[1].plot(history.history['val_loss'], linewidth=2.5, label='Validation', 
             color='#ff6600', marker='s', markersize=4, alpha=0.8)
axes[1].set_title('MobileNetV2 - Training vs Validation Loss', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Loss', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=12, loc='upper right')
axes[1].grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('mobilenetv2_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Training curves saved: mobilenetv2_training_curves.png")

# ============================================================================
# EVALUATION ON TEST SET
# ============================================================================
print_section("5. EVALUATION ON TEST SET")

print("\n🔍 Generating predictions on test set...")
test_gen.reset()
predictions = model.predict(test_gen, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_gen.classes

# Calculate overall accuracy
test_accuracy = accuracy_score(y_true, y_pred)
print(f"\n✓ Predictions generated for {len(y_true)} test samples")
print(f"  Test Set Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# ============================================================================
# METRICS TABLE
# ============================================================================
print_section("6. DETAILED CLASSIFICATION METRICS")

# Per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=range(num_classes)
)

# Create metrics dataframe
metrics_df = pd.DataFrame({
    'Class': class_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

# Add weighted average row
avg_precision, avg_recall, avg_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='weighted'
)

overall_row = pd.DataFrame({
    'Class': ['Overall (Weighted Avg)'],
    'Precision': [avg_precision],
    'Recall': [avg_recall],
    'F1-Score': [avg_f1],
    'Support': [len(y_true)]
})

metrics_df = pd.concat([metrics_df, overall_row], ignore_index=True)

print("\n📊 CLASSIFICATION PERFORMANCE METRICS (TEST SET):")
print(metrics_df.to_string(index=False))

# Save metrics to CSV
metrics_df.to_csv('mobilenetv2_test_metrics.csv', index=False)
print("\n✓ Metrics saved: mobilenetv2_test_metrics.csv")

# ============================================================================
# CLASSIFICATION REPORT
# ============================================================================
print("\n" + "-"*100)
print("DETAILED CLASSIFICATION REPORT")
print("-"*100)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ============================================================================
# CONFUSION MATRIX
# ============================================================================
print_section("7. CONFUSION MATRIX")

cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names, 
    yticklabels=class_names,
    cbar_kws={'label': 'Count'},
    annot_kws={'size': 14, 'weight': 'bold'},
    linewidths=1,
    linecolor='gray'
)
plt.title('Confusion Matrix - MobileNetV2 (Test Set)', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('Predicted Label', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=14, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold', rotation=45, ha='right')
plt.yticks(fontsize=12, fontweight='bold', rotation=0)
plt.tight_layout()
plt.savefig('mobilenetv2_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix saved: mobilenetv2_confusion_matrix.png")

# ============================================================================
# TIMING SUMMARY
# ============================================================================
print_section("8. EXECUTION TIME SUMMARY")

total_time = time.time() - start_time_total

print(f"\n⏱️  TIMING BREAKDOWN:")
print(f"  Training time: {format_time(train_time)}")
print(f"  Total pipeline time: {format_time(total_time)}")

# ============================================================================
# PREPARE RESULTS FOR XAI PIPELINES
# ============================================================================
print_section("9. PREPARING DATA FOR XAI EXPLAINABILITY ANALYSIS")

print("\n📋 Creating per-image classification results...")

# Get filepaths from test generator
filepaths = np.array(test_gen.filepaths)

# Create detailed results dataframe
results_df = pd.DataFrame({
    'filepath': filepaths,
    'TrueClassIdx': y_true,
    'PredClassIdx': y_pred,
    'TrueClass': [idx_to_class[i] for i in y_true],
    'PredClass': [idx_to_class[i] for i in y_pred],
    'MaxConfidence': np.max(predictions, axis=1),
    'Correct': y_true == y_pred
})

# Save full results
results_df.to_csv('mob_classification_test_results.csv', index=False)
print(f"✓ Full classification results: mob_classification_test_results.csv")
print(f"  ({len(results_df)} test samples)")

# ============================================================================
# SAMPLE SELECTION FOR XAI
# ============================================================================
print("\n🔍 Selecting samples for explainability analysis...")

# Select correctly classified samples (max 4 per class)
correct_samples = results_df[results_df['Correct']].groupby('TrueClass').apply(
    lambda x: x.head(4)
).reset_index(drop=True)

# Select misclassified samples (max 4 per class)
incorrect_samples = results_df[~results_df['Correct']].groupby('TrueClass').apply(
    lambda x: x.head(4)
).reset_index(drop=True)

correct_samples.to_csv('correct_XAI_selection.csv', index=False)
incorrect_samples.to_csv('misclassified_XAI_selection.csv', index=False)

print(f"✓ Correct samples selected: correct_XAI_selection.csv")
print(f"  ({len(correct_samples)} samples)")
print(f"✓ Misclassified samples selected: misclassified_XAI_selection.csv")
print(f"  ({len(incorrect_samples)} samples)")

# Print sample distribution
print("\n  Sample distribution by class:")
print("\n  CORRECTLY CLASSIFIED:")
print(correct_samples.groupby('TrueClass').size())
print("\n  MISCLASSIFIED:")
print(incorrect_samples.groupby('TrueClass').size())

# ============================================================================
# SAVE MODEL & METADATA
# ============================================================================
print_section("10. SAVING MODEL AND METADATA")

print("\n💾 Exporting model and metadata for XAI pipelines...")

# Save model
model.save('mob_model.h5')
print("✓ Model saved: mob_model.h5")

# Save class information
joblib.dump(class_names, 'mob_class_names.joblib')
joblib.dump(idx_to_class, 'mob_idx_to_class.joblib')
print("✓ Class information saved:")
print("  - mob_class_names.joblib")
print("  - mob_idx_to_class.joblib")

# ============================================================================
# SUMMARY REPORT
# ============================================================================
print_section("11. FINAL SUMMARY REPORT")

print(f"\n{'='*100}")
print(f"{'MOBILENETV2 TRAINING & EVALUATION SUMMARY':^100}")
print(f"{'='*100}")

summary_data = {
    'Metric': [
        'Total Parameters',
        'Training Samples',
        'Validation Samples',
        'Test Samples',
        'Epochs Trained',
        'Training Time',
        'Test Accuracy',
        'Weighted Precision',
        'Weighted Recall',
        'Weighted F1-Score',
        'Total Pipeline Time'
    ],
    'Value': [
        f'{total_params:,}',
        f'{train_gen.samples}',
        f'{val_gen.samples}',
        f'{test_gen.samples}',
        f'{NUM_EPOCHS}',
        format_time(train_time),
        f'{test_accuracy:.4f} ({test_accuracy*100:.2f}%)',
        f'{avg_precision:.4f}',
        f'{avg_recall:.4f}',
        f'{avg_f1:.4f}',
        format_time(total_time)
    ]
}

summary_df = pd.DataFrame(summary_data)
print(f"\n{summary_df.to_string(index=False)}")

print(f"\n{'='*100}")
print(f"{'SAVED FILES FOR XAI PIPELINES':^100}")
print(f"{'='*100}")
print("""
✓ MODEL AND METADATA:
  - mob_model.h5                          (Trained MobileNetV2 model)
  - mob_class_names.joblib                (Class name mapping)
  - mob_idx_to_class.joblib               (Index to class mapping)

✓ CLASSIFICATION RESULTS:
  - mob_classification_test_results.csv   (Per-image predictions & ground truth)
  - mobilenetv2_test_metrics.csv          (Performance metrics by class)

✓ XAI SAMPLE SELECTION:
  - correct_XAI_selection.csv             (Correctly classified samples for XAI)
  - misclassified_XAI_selection.csv       (Misclassified samples for XAI)

✓ VISUALIZATIONS:
  - mobilenetv2_training_curves.png       (Train/val accuracy & loss curves)
  - mobilenetv2_confusion_matrix.png      (Confusion matrix heatmap)
""")

print(f"\n{'='*100}")
print(f"✅ MobileNetV2 Pipeline Complete! Ready for XAI Analysis (LIME/GradCAM/RISE/SHAP)")
print(f"{'='*100}\n")

In [ ]:
!pip install lime
from lime import lime_image

In [ ]:
# ============================================================================
# SNIPPET 2: LIME Image Explainability - FIXED VERSION
# ============================================================================
# FIX: Use correct image size (250, 250) matching MobileNetV2 training
# ============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model
from lime import lime_image
from skimage.segmentation import mark_boundaries

# ======================== SETUP ========================
print("\n" + "="*100)
print("LOADING MobileNetV2 MODEL AND DATA")
print("="*100)

# Load model and metadata from Snippet 1
model_mob = load_model('mob_model.h5')
class_names = joblib.load('mob_class_names.joblib')
idx_to_class = joblib.load('mob_idx_to_class.joblib')

# Load test results and XAI sample selection CSVs
df_results = pd.read_csv('mob_classification_test_results.csv')
correct_samples = pd.read_csv('correct_XAI_selection.csv')
misclassified_samples = pd.read_csv('misclassified_XAI_selection.csv')

# *** CRITICAL FIX: Use correct image size matching MobileNetV2 training ***
IMG_SIZE = (250, 250)  # NOT (224, 224) - must match model training size
NUM_CLASSES = 4

print(f"✓ Model loaded successfully")
print(f"✓ Image size set to: {IMG_SIZE}")
print(f"✓ Number of classes: {NUM_CLASSES}")
print(f"✓ Correct samples: {len(correct_samples)}")
print(f"✓ Misclassified samples: {len(misclassified_samples)}")

# ======================== LIME EXPLAINER SETUP ========================
print("\n" + "="*100)
print("INITIALIZING LIME EXPLAINER")
print("="*100)

def preprocess_image_lime(image_path, target_size=IMG_SIZE):
    """Load image as RGB array (no batch dimension) - FIXED SIZE"""
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img)
    return img_array

def model_predict_lime(images):
    """Predictions for LIME (batch of images, normalized)"""
    # Ensure images are in correct shape
    if len(images.shape) == 3:
        images = np.expand_dims(images, axis=0)
    images_rescaled = images / 255.0
    return model_mob.predict(images_rescaled, verbose=0)

# Initialize LIME explainer
explainer_lime = lime_image.LimeImageExplainer()
print("✓ LIME explainer initialized")

# ======================== PROCESS CORRECTLY CLASSIFIED SAMPLES ========================
print("\n" + "="*100)
print("XAI METHOD 1: LIME - GROUP 1: CORRECTLY CLASSIFIED SAMPLES")
print("="*100)

lime_correct_results = []

for true_class in class_names:
    print(f"\n{'='*100}")
    print(f"CLASS: {true_class.upper()}")
    print(f"{'='*100}")
    
    class_samples = correct_samples[correct_samples['TrueClass'] == true_class]
    
    if len(class_samples) == 0:
        print(f"⚠️  No correctly classified samples for class {true_class}")
        continue
    
    for idx, (_, row) in enumerate(class_samples.iterrows()):
        img_path = row['filepath']
        true_class_idx = row['TrueClassIdx']
        pred_class_idx = row['PredClassIdx']
        
        if not os.path.exists(img_path):
            print(f"❌ Image not found: {img_path}")
            continue
        
        print(f"\n  [{idx+1}/{len(class_samples)}] {os.path.basename(img_path)}")
        
        try:
            # Load image with CORRECT SIZE
            img_array = preprocess_image_lime(img_path, target_size=IMG_SIZE)
            
            # Get model prediction - FIXED: pass image directly with correct shape
            img_batch = np.expand_dims(img_array, axis=0)
            pred_prob = model_mob.predict(img_batch / 255.0, verbose=0)
            pred_idx = np.argmax(pred_prob[0])
            pred_confidence = pred_prob[0][pred_idx]
            
            print(f"      True: {true_class} | Pred: {idx_to_class[pred_idx]} | Confidence: {pred_confidence:.4f}")
            
            # Generate LIME explanation
            explanation = explainer_lime.explain_instance(
                img_array.astype('double'),
                model_predict_lime,
                top_labels=1,
                hide_color=0,
                num_samples=1000,
                random_seed=42
            )
            
            # Get image and mask
            temp, mask = explanation.get_image_and_mask(
                pred_idx,
                positive_only=True,
                num_features=5,
                hide_rest=False
            )
            
            # Calculate metrics for the explanation
            num_superpixels = len(np.unique(mask))
            avg_feature_weight = np.mean(explanation.local_exp[pred_idx]) if explanation.local_exp[pred_idx] else 0
            
            # ===== Display Original Image =====
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(img_array.astype('uint8'))
            plt.title(f"{true_class.upper()} - Original Image\n{os.path.basename(img_path)}", 
                     fontsize=12, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # ===== Display LIME Explanation =====
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(mark_boundaries(temp / 255.0, mask))
            title_text = (
                f"LIME - {true_class.upper()} (CORRECT PREDICTION)\n"
                f"Pred: {idx_to_class[pred_idx]} | Conf: {pred_confidence:.4f}\n"
                f"Superpixels: {num_superpixels} | Avg Weight: {avg_feature_weight:.4f}"
            )
            plt.title(title_text, fontsize=11, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # Store results
            lime_correct_results.append({
                'Class': true_class,
                'Filename': os.path.basename(img_path),
                'Filepath': img_path,
                'TrueClassIdx': true_class_idx,
                'PredClassIdx': pred_idx,
                'Confidence': pred_confidence,
                'Method': 'LIME',
                'Group': 'Correct',
                'Num_Superpixels': num_superpixels,
                'Avg_Feature_Weight': avg_feature_weight
            })
            
        except Exception as e:
            print(f"❌ Error processing {os.path.basename(img_path)}: {str(e)}")

# ======================== PROCESS MISCLASSIFIED SAMPLES ========================
print("\n\n" + "="*100)
print("XAI METHOD 1: LIME - GROUP 2: MISCLASSIFIED SAMPLES")
print("="*100)

lime_incorrect_results = []

for true_class in class_names:
    print(f"\n{'='*100}")
    print(f"CLASS: {true_class.upper()}")
    print(f"{'='*100}")
    
    class_samples = misclassified_samples[misclassified_samples['TrueClass'] == true_class]
    
    if len(class_samples) == 0:
        print(f"⚠️  No misclassified samples for class {true_class}")
        continue
    
    for idx, (_, row) in enumerate(class_samples.iterrows()):
        img_path = row['filepath']
        true_class_idx = row['TrueClassIdx']
        pred_class_idx = row['PredClassIdx']
        
        if not os.path.exists(img_path):
            print(f"❌ Image not found: {img_path}")
            continue
        
        print(f"\n  [{idx+1}/{len(class_samples)}] {os.path.basename(img_path)}")
        
        try:
            # Load image with CORRECT SIZE
            img_array = preprocess_image_lime(img_path, target_size=IMG_SIZE)
            
            # Get model prediction - FIXED: pass image directly with correct shape
            img_batch = np.expand_dims(img_array, axis=0)
            pred_prob = model_mob.predict(img_batch / 255.0, verbose=0)
            pred_idx = np.argmax(pred_prob[0])
            pred_confidence = pred_prob[0][pred_idx]
            
            print(f"      True: {true_class} | Pred: {idx_to_class[pred_idx]} (WRONG) | Confidence: {pred_confidence:.4f}")
            
            # Generate LIME explanation
            explanation = explainer_lime.explain_instance(
                img_array.astype('double'),
                model_predict_lime,
                top_labels=1,
                hide_color=0,
                num_samples=1000,
                random_seed=42
            )
            
            # Get image and mask
            temp, mask = explanation.get_image_and_mask(
                pred_idx,
                positive_only=True,
                num_features=5,
                hide_rest=False
            )
            
            # Calculate metrics
            num_superpixels = len(np.unique(mask))
            avg_feature_weight = np.mean(explanation.local_exp[pred_idx]) if explanation.local_exp[pred_idx] else 0
            
            # ===== Display Original Image =====
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(img_array.astype('uint8'))
            plt.title(f"{true_class.upper()} - Original Image (Misclassified)\n{os.path.basename(img_path)}", 
                     fontsize=12, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # ===== Display LIME Explanation =====
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(mark_boundaries(temp / 255.0, mask))
            title_text = (
                f"LIME - {true_class.upper()} (MISCLASSIFIED)\n"
                f"True: {true_class} | Pred: {idx_to_class[pred_idx]} | Conf: {pred_confidence:.4f}\n"
                f"Superpixels: {num_superpixels} | Avg Weight: {avg_feature_weight:.4f}"
            )
            plt.title(title_text, fontsize=11, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # Store results
            lime_incorrect_results.append({
                'Class': true_class,
                'Filename': os.path.basename(img_path),
                'Filepath': img_path,
                'TrueClassIdx': true_class_idx,
                'PredClassIdx': pred_idx,
                'Confidence': pred_confidence,
                'Method': 'LIME',
                'Group': 'Misclassified',
                'Num_Superpixels': num_superpixels,
                'Avg_Feature_Weight': avg_feature_weight
            })
            
        except Exception as e:
            print(f"❌ Error processing {os.path.basename(img_path)}: {str(e)}")

# ======================== SAVE RESULTS ========================
lime_all_results = lime_correct_results + lime_incorrect_results
lime_df = pd.DataFrame(lime_all_results)
lime_df.to_csv('LIME_explainability_results.csv', index=False)

print("\n" + "="*100)
print("✓ LIME EXPLAINABILITY ANALYSIS COMPLETE")
print("="*100)
print(f"  Total explanations generated: {len(lime_df)}")
print(f"  Results saved to: LIME_explainability_results.csv")
print(f"\n  Breakdown:")
print(f"    Correctly classified: {len([x for x in lime_all_results if x['Group']=='Correct'])}")
print(f"    Misclassified: {len([x for x in lime_all_results if x['Group']=='Misclassified'])}")
for cls in class_names:
    cls_count = len([x for x in lime_all_results if x['Class']==cls])
    print(f"    {cls}: {cls_count}")
print("="*100 + "\n")

In [ ]:
# ============================================================================
# SNIPPET 3: Grad-CAM Image Explainability - FIXED VERSION
# ============================================================================
# FIX: Use correct image size (250, 250) matching MobileNetV2 training
# ============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
import keras
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model

# ======================== SETUP ========================
print("\n" + "="*100)
print("LOADING MobileNetV2 MODEL AND DATA")
print("="*100)

model_mob = load_model('mob_model.h5')
class_names = joblib.load('mob_class_names.joblib')
idx_to_class = joblib.load('mob_idx_to_class.joblib')

df_results = pd.read_csv('mob_classification_test_results.csv')
correct_samples = pd.read_csv('correct_XAI_selection.csv')
misclassified_samples = pd.read_csv('misclassified_XAI_selection.csv')

# *** CRITICAL FIX: Use correct image size matching MobileNetV2 training ***
IMG_SIZE = (250, 250)  # NOT (224, 224) - must match model training size
NUM_CLASSES = 4

print(f"✓ Model loaded successfully")
print(f"✓ Image size set to: {IMG_SIZE}")
print(f"✓ Number of classes: {NUM_CLASSES}")
print(f"✓ Correct samples: {len(correct_samples)}")
print(f"✓ Misclassified samples: {len(misclassified_samples)}")

# ======================== GRAD-CAM SETUP ========================
print("\n" + "="*100)
print("INITIALIZING GRAD-CAM EXPLAINER")
print("="*100)

def get_last_conv_layer(model):
    """Get the last convolutional layer name"""
    last_conv_layer = None
    for layer in reversed(model.layers):
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
            last_conv_layer = layer.name
            break
    return last_conv_layer

last_conv_layer_name = get_last_conv_layer(model_mob)
print(f"✓ Last convolutional layer identified: {last_conv_layer_name}")

def get_img_array(img_path, size=IMG_SIZE):
    """Load and preprocess image for Grad-CAM - FIXED SIZE"""
    img = load_img(img_path, target_size=size)
    array = img_to_array(img)
    # Add batch dimension
    array = np.expand_dims(array, axis=0)
    return array

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Generate Grad-CAM heatmap"""
    grad_model = tf.keras.models.Model(
        model.inputs, [model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    
    return heatmap.numpy()

def display_gradcam_overlay(img_path, heatmap, alpha=0.4):
    """Create overlay of Grad-CAM on original image"""
    # Load original image with CORRECT SIZE
    img = img_to_array(load_img(img_path, target_size=IMG_SIZE))
    
    # Rescale heatmap
    heatmap = np.uint8(255 * heatmap)
    
    # Apply jet colormap
    import matplotlib.cm as cm
    jet = cm.get_cmap('jet')
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    jet_heatmap = keras.utils.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = keras.utils.img_to_array(jet_heatmap)
    
    # Superimpose
    superimposed_img = jet_heatmap * alpha + img
    superimposed_img = keras.utils.array_to_img(superimposed_img)
    
    return superimposed_img

print("✓ Grad-CAM explainer initialized")

# ======================== PROCESS CORRECTLY CLASSIFIED SAMPLES ========================
print("\n" + "="*100)
print("XAI METHOD 2: GRAD-CAM - GROUP 1: CORRECTLY CLASSIFIED SAMPLES")
print("="*100)

gradcam_correct_results = []

for true_class in class_names:
    print(f"\n{'='*100}")
    print(f"CLASS: {true_class.upper()}")
    print(f"{'='*100}")
    
    class_samples = correct_samples[correct_samples['TrueClass'] == true_class]
    
    if len(class_samples) == 0:
        print(f"⚠️  No correctly classified samples for class {true_class}")
        continue
    
    for idx, (_, row) in enumerate(class_samples.iterrows()):
        img_path = row['filepath']
        true_class_idx = row['TrueClassIdx']
        pred_class_idx = row['PredClassIdx']
        
        if not os.path.exists(img_path):
            print(f"❌ Image not found: {img_path}")
            continue
        
        print(f"\n  [{idx+1}/{len(class_samples)}] {os.path.basename(img_path)}")
        
        try:
            # Get image array with CORRECT SIZE
            img_array = get_img_array(img_path, size=IMG_SIZE)
            
            # Get prediction - FIXED: normalize properly
            pred_prob = model_mob.predict(img_array / 255.0, verbose=0)
            pred_idx = np.argmax(pred_prob[0])
            pred_confidence = pred_prob[0][pred_idx]
            
            print(f"      True: {true_class} | Pred: {idx_to_class[pred_idx]} | Confidence: {pred_confidence:.4f}")
            
            # Generate Grad-CAM heatmap
            heatmap = make_gradcam_heatmap(img_array / 255.0, model_mob, last_conv_layer_name, pred_index=pred_idx)
            
            # Calculate metrics
            heatmap_max = np.max(heatmap)
            heatmap_mean = np.mean(heatmap)
            heatmap_std = np.std(heatmap)
            
            # ===== Display Original Image =====
            fig = plt.figure(figsize=(6, 6))
            img_original = load_img(img_path, target_size=IMG_SIZE)
            plt.imshow(img_original)
            plt.title(f"{true_class.upper()} - Original Image\n{os.path.basename(img_path)}", 
                     fontsize=12, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # ===== Display Grad-CAM Heatmap =====
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(heatmap, cmap='jet')
            plt.colorbar(fraction=0.046, pad=0.04, label='Saliency')
            title_text = (
                f"Grad-CAM - {true_class.upper()} (CORRECT)\n"
                f"Max: {heatmap_max:.4f} | Mean: {heatmap_mean:.4f} | Std: {heatmap_std:.4f}"
            )
            plt.title(title_text, fontsize=11, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # ===== Display Overlay =====
            fig = plt.figure(figsize=(6, 6))
            overlay = display_gradcam_overlay(img_path, heatmap, alpha=0.4)
            plt.imshow(overlay)
            title_text = (
                f"Grad-CAM Overlay - {true_class.upper()}\n"
                f"Pred: {idx_to_class[pred_idx]} | Confidence: {pred_confidence:.4f}"
            )
            plt.title(title_text, fontsize=11, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # Store results
            gradcam_correct_results.append({
                'Class': true_class,
                'Filename': os.path.basename(img_path),
                'Filepath': img_path,
                'TrueClassIdx': true_class_idx,
                'PredClassIdx': pred_idx,
                'Confidence': pred_confidence,
                'Method': 'Grad-CAM',
                'Group': 'Correct',
                'Heatmap_Max': heatmap_max,
                'Heatmap_Mean': heatmap_mean,
                'Heatmap_Std': heatmap_std
            })
            
        except Exception as e:
            print(f"❌ Error processing {os.path.basename(img_path)}: {str(e)}")

# ======================== PROCESS MISCLASSIFIED SAMPLES ========================
print("\n\n" + "="*100)
print("XAI METHOD 2: GRAD-CAM - GROUP 2: MISCLASSIFIED SAMPLES")
print("="*100)

gradcam_incorrect_results = []

for true_class in class_names:
    print(f"\n{'='*100}")
    print(f"CLASS: {true_class.upper()}")
    print(f"{'='*100}")
    
    class_samples = misclassified_samples[misclassified_samples['TrueClass'] == true_class]
    
    if len(class_samples) == 0:
        print(f"⚠️  No misclassified samples for class {true_class}")
        continue
    
    for idx, (_, row) in enumerate(class_samples.iterrows()):
        img_path = row['filepath']
        true_class_idx = row['TrueClassIdx']
        pred_class_idx = row['PredClassIdx']
        
        if not os.path.exists(img_path):
            print(f"❌ Image not found: {img_path}")
            continue
        
        print(f"\n  [{idx+1}/{len(class_samples)}] {os.path.basename(img_path)}")
        
        try:
            # Get image array with CORRECT SIZE
            img_array = get_img_array(img_path, size=IMG_SIZE)
            
            # Get prediction - FIXED: normalize properly
            pred_prob = model_mob.predict(img_array / 255.0, verbose=0)
            pred_idx = np.argmax(pred_prob[0])
            pred_confidence = pred_prob[0][pred_idx]
            
            print(f"      True: {true_class} | Pred: {idx_to_class[pred_idx]} (WRONG) | Confidence: {pred_confidence:.4f}")
            
            # Generate Grad-CAM heatmap
            heatmap = make_gradcam_heatmap(img_array / 255.0, model_mob, last_conv_layer_name, pred_index=pred_idx)
            
            # Calculate metrics
            heatmap_max = np.max(heatmap)
            heatmap_mean = np.mean(heatmap)
            heatmap_std = np.std(heatmap)
            
            # ===== Display Original Image =====
            fig = plt.figure(figsize=(6, 6))
            img_original = load_img(img_path, target_size=IMG_SIZE)
            plt.imshow(img_original)
            plt.title(f"{true_class.upper()} - Original Image (Misclassified)\n{os.path.basename(img_path)}", 
                     fontsize=12, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # ===== Display Grad-CAM Heatmap =====
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(heatmap, cmap='jet')
            plt.colorbar(fraction=0.046, pad=0.04, label='Saliency')
            title_text = (
                f"Grad-CAM - {true_class.upper()} (MISCLASSIFIED)\n"
                f"Max: {heatmap_max:.4f} | Mean: {heatmap_mean:.4f} | Std: {heatmap_std:.4f}"
            )
            plt.title(title_text, fontsize=11, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # ===== Display Overlay =====
            fig = plt.figure(figsize=(6, 6))
            overlay = display_gradcam_overlay(img_path, heatmap, alpha=0.4)
            plt.imshow(overlay)
            title_text = (
                f"Grad-CAM Overlay - {true_class.upper()}\n"
                f"True: {true_class} | Pred: {idx_to_class[pred_idx]} | Confidence: {pred_confidence:.4f}"
            )
            plt.title(title_text, fontsize=11, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # Store results
            gradcam_incorrect_results.append({
                'Class': true_class,
                'Filename': os.path.basename(img_path),
                'Filepath': img_path,
                'TrueClassIdx': true_class_idx,
                'PredClassIdx': pred_idx,
                'Confidence': pred_confidence,
                'Method': 'Grad-CAM',
                'Group': 'Misclassified',
                'Heatmap_Max': heatmap_max,
                'Heatmap_Mean': heatmap_mean,
                'Heatmap_Std': heatmap_std
            })
            
        except Exception as e:
            print(f"❌ Error processing {os.path.basename(img_path)}: {str(e)}")

# ======================== SAVE RESULTS ========================
gradcam_all_results = gradcam_correct_results + gradcam_incorrect_results
gradcam_df = pd.DataFrame(gradcam_all_results)
gradcam_df.to_csv('GradCAM_explainability_results.csv', index=False)

print("\n" + "="*100)
print("✓ GRAD-CAM EXPLAINABILITY ANALYSIS COMPLETE")
print("="*100)
print(f"  Total explanations generated: {len(gradcam_df)}")
print(f"  Results saved to: GradCAM_explainability_results.csv")
print(f"\n  Breakdown:")
print(f"    Correctly classified: {len([x for x in gradcam_all_results if x['Group']=='Correct'])}")
print(f"    Misclassified: {len([x for x in gradcam_all_results if x['Group']=='Misclassified'])}")
for cls in class_names:
    cls_count = len([x for x in gradcam_all_results if x['Class']==cls])
    print(f"    {cls}: {cls_count}")
print("="*100 + "\n")

In [ ]:
# ============================================================================
# SNIPPET 4: RISE Image Explainability - FIXED VERSION
# ============================================================================
# FIX: Use correct image size (250, 250) matching MobileNetV2 training
# ============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import cv2
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model
from tqdm import tqdm

# ======================== SETUP ========================
print("\n" + "="*100)
print("LOADING MobileNetV2 MODEL AND DATA")
print("="*100)

model_mob = load_model('mob_model.h5')
class_names = joblib.load('mob_class_names.joblib')
idx_to_class = joblib.load('mob_idx_to_class.joblib')

df_results = pd.read_csv('mob_classification_test_results.csv')
correct_samples = pd.read_csv('correct_XAI_selection.csv')
misclassified_samples = pd.read_csv('misclassified_XAI_selection.csv')

# *** CRITICAL FIX: Use correct image size matching MobileNetV2 training ***
IMG_SIZE = (250, 250)  # NOT (224, 224) - must match model training size
NUM_CLASSES = 4

print(f"✓ Model loaded successfully")
print(f"✓ Image size set to: {IMG_SIZE}")
print(f"✓ Number of classes: {NUM_CLASSES}")
print(f"✓ Correct samples: {len(correct_samples)}")
print(f"✓ Misclassified samples: {len(misclassified_samples)}")

# ======================== RISE EXPLAINER CLASS ========================
print("\n" + "="*100)
print("INITIALIZING RISE EXPLAINER")
print("="*100)

class RISE:
    """RISE (Randomized Input Sampling for Explanation) explainer"""
    
    def __init__(self, model, input_size, gpu_batch=50):
        self.model = model
        self.input_size = input_size
        self.gpu_batch = gpu_batch
        self.masks = None
        self.p1 = None
    
    def generate_masks(self, N=2000, s=8, p1=0.1):
        """Generate random masks for RISE"""
        print(f"\nGenerating {N} RISE masks (grid: {s}x{s}, p1: {p1})...")
        
        cell_size = np.ceil(np.array(self.input_size) / s).astype(int)
        up_size = (s + 1) * cell_size
        
        grid = np.random.rand(N, s, s) < p1
        grid = grid.astype('float32')
        
        self.masks = np.empty((N, *self.input_size))
        
        for i in tqdm(range(N), desc="Generating masks", leave=True):
            x = np.random.randint(0, cell_size[0])
            y = np.random.randint(0, cell_size[1])
            
            # Resize grid to up_size
            mask = cv2.resize(
                grid[i], 
                (up_size[1], up_size[0]), 
                interpolation=cv2.INTER_LINEAR
            )
            
            # Crop to input_size
            mask = mask[x:x + self.input_size[0], y:y + self.input_size[1]]
            
            # Handle edge cases with padding
            if mask.shape[0] < self.input_size[0] or mask.shape[1] < self.input_size[1]:
                pad_h = self.input_size[0] - mask.shape[0]
                pad_w = self.input_size[1] - mask.shape[1]
                mask = np.pad(mask, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0)
            
            self.masks[i] = mask[:self.input_size[0], :self.input_size[1]]
        
        # Add channel dimension
        self.masks = self.masks.reshape(-1, *self.input_size, 1)
        self.p1 = p1
        print(f"✓ Generated {N} masks with shape: {self.masks.shape}")
    
    def __call__(self, input_image):
        """Compute RISE saliency maps for input image"""
        N = self.masks.shape[0]
        saliency = np.zeros((NUM_CLASSES, *self.input_size))
        
        print(f"\nComputing RISE saliency maps for image...")
        
        for i in tqdm(range(0, N, self.gpu_batch), desc="RISE computation", leave=True):
            batch_end = min(i + self.gpu_batch, N)
            batch_masks = self.masks[i:batch_end]
            
            # Apply masks to input image
            masked_inputs = input_image * batch_masks
            
            # Get predictions for masked images
            preds = self.model.predict(masked_inputs, verbose=0)
            
            # Accumulate saliency for each class
            for j, pred in enumerate(preds):
                for c in range(NUM_CLASSES):
                    saliency[c] += pred[c] * batch_masks[j, :, :, 0]
        
        # Normalize saliency
        saliency = saliency / N / self.p1
        return saliency

# Initialize RISE
rise_explainer = RISE(model_mob, input_size=IMG_SIZE, gpu_batch=50)
rise_explainer.generate_masks(N=2000, s=8, p1=0.1)

print("✓ RISE explainer initialized")

# ======================== PROCESS CORRECTLY CLASSIFIED SAMPLES ========================
print("\n" + "="*100)
print("XAI METHOD 3: RISE - GROUP 1: CORRECTLY CLASSIFIED SAMPLES")
print("="*100)

rise_correct_results = []

for true_class in class_names:
    print(f"\n{'='*100}")
    print(f"CLASS: {true_class.upper()}")
    print(f"{'='*100}")
    
    class_samples = correct_samples[correct_samples['TrueClass'] == true_class]
    
    if len(class_samples) == 0:
        print(f"⚠️  No correctly classified samples for class {true_class}")
        continue
    
    for idx, (_, row) in enumerate(class_samples.iterrows()):
        img_path = row['filepath']
        true_class_idx = row['TrueClassIdx']
        pred_class_idx = row['PredClassIdx']
        
        if not os.path.exists(img_path):
            print(f"❌ Image not found: {img_path}")
            continue
        
        print(f"\n  [{idx+1}/{len(class_samples)}] {os.path.basename(img_path)}")
        
        try:
            # Load image with CORRECT SIZE
            img = load_img(img_path, target_size=IMG_SIZE)
            img_array = img_to_array(img)
            
            # Create batch and normalize
            img_batch = np.expand_dims(img_array / 255.0, axis=0)
            
            # Get prediction
            pred_prob = model_mob.predict(img_batch, verbose=0)
            pred_idx = np.argmax(pred_prob[0])
            pred_confidence = pred_prob[0][pred_idx]
            
            print(f"      True: {true_class} | Pred: {idx_to_class[pred_idx]} | Confidence: {pred_confidence:.4f}")
            
            # Generate RISE saliency maps
            saliency_maps = rise_explainer(img_batch)
            
            # Normalize saliency for all classes
            saliencies_normalized = {}
            for c in range(NUM_CLASSES):
                sal = saliency_maps[c]
                sal_min = sal.min()
                sal_max = sal.max()
                sal_norm = (sal - sal_min) / (sal_max - sal_min + 1e-8)
                saliencies_normalized[c] = sal_norm
            
            # ===== Display Original Image =====
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(img)
            plt.title(f"{true_class.upper()} - Original Image\n{os.path.basename(img_path)}", 
                     fontsize=12, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # ===== Display RISE Saliency for Predicted Class =====
            sal_pred = saliencies_normalized[pred_idx]
            sal_max = np.max(sal_pred)
            sal_mean = np.mean(sal_pred)
            
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(sal_pred, cmap='jet')
            plt.colorbar(fraction=0.046, pad=0.04, label='Saliency')
            title_text = (
                f"RISE - {idx_to_class[pred_idx].upper()} (CORRECT)\n"
                f"Max: {sal_max:.4f} | Mean: {sal_mean:.4f}"
            )
            plt.title(title_text, fontsize=11, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # ===== Display RISE Overlay =====
            img_np = np.array(img)
            sal_resized = cv2.resize(sal_pred, (img_np.shape[1], img_np.shape[0]))
            sal_heatmap = np.uint8(255 * sal_resized)
            sal_heatmap_color = cv2.applyColorMap(sal_heatmap, cv2.COLORMAP_JET)
            overlay = cv2.addWeighted(sal_heatmap_color, 0.5, img_np, 0.5, 0)
            
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
            title_text = (
                f"RISE Overlay - {true_class.upper()}\n"
                f"Pred: {idx_to_class[pred_idx]} | Confidence: {pred_confidence:.4f}"
            )
            plt.title(title_text, fontsize=11, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # Store results
            rise_correct_results.append({
                'Class': true_class,
                'Filename': os.path.basename(img_path),
                'Filepath': img_path,
                'TrueClassIdx': true_class_idx,
                'PredClassIdx': pred_idx,
                'Confidence': pred_confidence,
                'Method': 'RISE',
                'Group': 'Correct',
                'Saliency_Max': sal_max,
                'Saliency_Mean': sal_mean
            })
            
        except Exception as e:
            print(f"❌ Error processing {os.path.basename(img_path)}: {str(e)}")

# ======================== PROCESS MISCLASSIFIED SAMPLES ========================
print("\n\n" + "="*100)
print("XAI METHOD 3: RISE - GROUP 2: MISCLASSIFIED SAMPLES")
print("="*100)

rise_incorrect_results = []

for true_class in class_names:
    print(f"\n{'='*100}")
    print(f"CLASS: {true_class.upper()}")
    print(f"{'='*100}")
    
    class_samples = misclassified_samples[misclassified_samples['TrueClass'] == true_class]
    
    if len(class_samples) == 0:
        print(f"⚠️  No misclassified samples for class {true_class}")
        continue
    
    for idx, (_, row) in enumerate(class_samples.iterrows()):
        img_path = row['filepath']
        true_class_idx = row['TrueClassIdx']
        pred_class_idx = row['PredClassIdx']
        
        if not os.path.exists(img_path):
            print(f"❌ Image not found: {img_path}")
            continue
        
        print(f"\n  [{idx+1}/{len(class_samples)}] {os.path.basename(img_path)}")
        
        try:
            # Load image with CORRECT SIZE
            img = load_img(img_path, target_size=IMG_SIZE)
            img_array = img_to_array(img)
            
            # Create batch and normalize
            img_batch = np.expand_dims(img_array / 255.0, axis=0)
            
            # Get prediction
            pred_prob = model_mob.predict(img_batch, verbose=0)
            pred_idx = np.argmax(pred_prob[0])
            pred_confidence = pred_prob[0][pred_idx]
            
            print(f"      True: {true_class} | Pred: {idx_to_class[pred_idx]} (WRONG) | Confidence: {pred_confidence:.4f}")
            
            # Generate RISE saliency maps
            saliency_maps = rise_explainer(img_batch)
            
            # Normalize saliency
            saliencies_normalized = {}
            for c in range(NUM_CLASSES):
                sal = saliency_maps[c]
                sal_min = sal.min()
                sal_max = sal.max()
                sal_norm = (sal - sal_min) / (sal_max - sal_min + 1e-8)
                saliencies_normalized[c] = sal_norm
            
            # ===== Display Original Image =====
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(img)
            plt.title(f"{true_class.upper()} - Original Image (Misclassified)\n{os.path.basename(img_path)}", 
                     fontsize=12, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # ===== Display RISE Saliency for Predicted Class =====
            sal_pred = saliencies_normalized[pred_idx]
            sal_max = np.max(sal_pred)
            sal_mean = np.mean(sal_pred)
            
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(sal_pred, cmap='jet')
            plt.colorbar(fraction=0.046, pad=0.04, label='Saliency')
            title_text = (
                f"RISE - {idx_to_class[pred_idx].upper()} (MISCLASSIFIED)\n"
                f"Max: {sal_max:.4f} | Mean: {sal_mean:.4f}"
            )
            plt.title(title_text, fontsize=11, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # ===== Display RISE Overlay =====
            img_np = np.array(img)
            sal_resized = cv2.resize(sal_pred, (img_np.shape[1], img_np.shape[0]))
            sal_heatmap = np.uint8(255 * sal_resized)
            sal_heatmap_color = cv2.applyColorMap(sal_heatmap, cv2.COLORMAP_JET)
            overlay = cv2.addWeighted(sal_heatmap_color, 0.5, img_np, 0.5, 0)
            
            fig = plt.figure(figsize=(6, 6))
            plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
            title_text = (
                f"RISE Overlay - {true_class.upper()}\n"
                f"True: {true_class} | Pred: {idx_to_class[pred_idx]} | Confidence: {pred_confidence:.4f}"
            )
            plt.title(title_text, fontsize=11, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            # Store results
            rise_incorrect_results.append({
                'Class': true_class,
                'Filename': os.path.basename(img_path),
                'Filepath': img_path,
                'TrueClassIdx': true_class_idx,
                'PredClassIdx': pred_idx,
                'Confidence': pred_confidence,
                'Method': 'RISE',
                'Group': 'Misclassified',
                'Saliency_Max': sal_max,
                'Saliency_Mean': sal_mean
            })
            
        except Exception as e:
            print(f"❌ Error processing {os.path.basename(img_path)}: {str(e)}")

# ======================== SAVE RESULTS ========================
rise_all_results = rise_correct_results + rise_incorrect_results
rise_df = pd.DataFrame(rise_all_results)
rise_df.to_csv('RISE_explainability_results.csv', index=False)

print("\n" + "="*100)
print("✓ RISE EXPLAINABILITY ANALYSIS COMPLETE")
print("="*100)
print(f"  Total explanations generated: {len(rise_df)}")
print(f"  Results saved to: RISE_explainability_results.csv")
print(f"\n  Breakdown:")
print(f"    Correctly classified: {len([x for x in rise_all_results if x['Group']=='Correct'])}")
print(f"    Misclassified: {len([x for x in rise_all_results if x['Group']=='Misclassified'])}")
for cls in class_names:
    cls_count = len([x for x in rise_all_results if x['Class']==cls])
    print(f"    {cls}: {cls_count}")
print("="*100 + "\n")